## 7. Part 8: expected price, price gap and flag status (Tân)

**Question:** for each listing, how far is its price per guest from what is typical for comparable listings, and is it flagged as *possibly overpriced*?

This section follows the group decisions (1 to 6) and runs after `df_clean` has been created in Section 1.

| Decision | What it means here |
|---|---|
| 1. Price measure | `price_per_guest` = `price / accommodates` (USD per guest per night) |
| 2. Comparison group | `neighbourhood_cleansed` × `room_type`. Bedrooms are **not** part of the group key |
| 3. Minimum group size | A comparison group needs at least 20 listings. Smaller groups use the room-type group instead (decision 4 fallback) |
| 4. Flagged | Price per guest is unusually high within its comparison group **and** not explained by amenities |
| 5. Statistic | Median (right-skewed data); the fence is computed on log10 of price per guest |

**Status rules (checked in this order, so the three statuses never overlap):**
1. `not enough data`: `bedrooms` is missing, so the listing cannot be compared with peers of the same size on amenities.
2. `flagged`: above the group fence AND `amenities_count` is not above the median of its peers (same room type and same `bedrooms_grp`, 4 means 4 or more).
3. `not flagged`: every other listing.

A flag marks an unusual price within a comparison group. It does not prove that the listing is overpriced.

In [ ]:
MIN_GROUP = 20            # decision 3
GROUP_KEYS = ["neighbourhood_cleansed", "room_type"]   # decision 2

p = df_clean.copy()
p["price_per_guest"] = p["price"] / p["accommodates"]     # decision 1
p["log_ppg"] = np.log10(p["price_per_guest"])
assert p["price_per_guest"].notna().all() and (p["price_per_guest"] > 0).all()
print("listings:", len(p))

### 7.1 Expected price and fence per comparison group

For every group (area × room type) we compute the median and the fence `Q3 + 1.5 x IQR` on log10 of price per guest. Groups with fewer than 20 listings take the values of their **room-type** group (decision 4 fallback).

In [ ]:
def group_stats(g):
    q1, q3 = g["log_ppg"].quantile(0.25), g["log_ppg"].quantile(0.75)
    return pd.Series({"median_log": g["log_ppg"].median(), "fence_log": q3 + 1.5 * (q3 - q1)})

area_stats = p.groupby(GROUP_KEYS).apply(group_stats, include_groups=False)
room_stats = p.groupby("room_type").apply(group_stats, include_groups=False)

p["group_size"] = p.groupby(GROUP_KEYS)["id"].transform("size")
p = p.join(area_stats, on=GROUP_KEYS)

# Fallback: groups with fewer than 20 listings use the room-type values
small = p["group_size"] < MIN_GROUP
p["used_fallback"] = small
for c in ["median_log", "fence_log"]:
    p.loc[small, c] = p.loc[small, "room_type"].map(room_stats[c])

p["expected_ppg"] = 10 ** p["median_log"]            # expected price per guest (median of the group)
p["fence_ppg"] = 10 ** p["fence_log"]
p["price_gap"] = p["price_per_guest"] - p["expected_ppg"]          # USD per guest per night
p["price_gap_pct"] = p["price_gap"] / p["expected_ppg"] * 100
p["above_fence"] = p["log_ppg"] > p["fence_log"]      # compare in log space so listings exactly on the fence are not flagged by rounding

n_groups = p.groupby(GROUP_KEYS).ngroups
n_small = p[small].groupby(GROUP_KEYS).ngroups
print(f"{n_groups} comparison groups; {n_small} have fewer than {MIN_GROUP} listings "
      f"({int(small.sum()):,} listings use the room-type fallback)")
print("above group fence:", int(p["above_fence"].sum()))

### 7.2 Amenities check and status

Peers for the amenities check are listings with the same room type and the same `bedrooms_grp`. A high price is *explained by amenities* when the listing has more amenities than the peer median.

In [ ]:
p["bedrooms_grp"] = p["bedrooms"].clip(upper=4)          # 4 means "4 or more"
p["peer_median_amenities"] = p.groupby(["room_type", "bedrooms_grp"])["amenities_count"].transform("median")
p["explained_by_amenities"] = p["amenities_count"] > p["peer_median_amenities"]

no_data = p["bedrooms"].isna()                                      # status 1
flagged = ~no_data & p["above_fence"] & ~p["explained_by_amenities"]   # status 2

p["status"] = np.select([no_data, flagged], ["not enough data", "flagged"], default="not flagged")

### 7.3 Three-status table (feeds Chart 4)

In [ ]:
order = ["flagged", "not flagged", "not enough data"]
F8_status_counts = (p["status"].value_counts().reindex(order).rename_axis("status")
                    .reset_index(name="listings"))
F8_status_counts["share_%"] = F8_status_counts["listings"] / len(p) * 100

# Checks: the statuses do not overlap and add up to all listings
assert F8_status_counts["listings"].sum() == len(p) == 51864
assert p["status"].notna().all()
F8_status_counts.round(2)

### 7.4 Checks and notes for the report

In [ ]:
print("Above group fence:", int(p["above_fence"].sum()))
print("  of which not enough data (no bedrooms):", int((p["above_fence"] & no_data).sum()))
print("  of which explained by amenities (not flagged):",
      int((p["above_fence"] & ~no_data & p["explained_by_amenities"]).sum()))
print("  of which flagged:", int(flagged.sum()))
print("Listings in groups below 20 (room-type fallback):", int(small.sum()))
print("Not enough data AND in a small group:", int((no_data & small).sum()))

# Flagged listings by room type and the largest gaps
print(p[flagged].groupby("room_type")["id"].size().rename("flagged_listings").to_frame())
p.loc[flagged, ["id", "neighbourhood_cleansed", "room_type", "bedrooms", "amenities_count",
                "peer_median_amenities", "price_per_guest", "expected_ppg", "price_gap",
                "price_gap_pct"]].sort_values("price_gap", ascending=False).head(10).round(2)

**Notes (for the cleaning log and report):**
- The fence is now computed per area × room type, so the number of listings above the fence differs from the 669 `price_outlier` flags in the cleaning log (step 14), which uses room type only. Decision 4 states this change.
- `not enough data` contains only listings with missing `bedrooms` (7,859, 15%). Small groups are not "not enough data" because they use the room-type fallback.
- Some listings with missing bedrooms are above their group fence. They are counted as `not enough data` because the amenities check needs the bedroom count.
- `amenities_count` measures quantity, not quality of amenities.

### 7.5 Save outputs

In [ ]:
F8_listing_flags = p[["id", "neighbourhood_cleansed", "room_type", "group_size", "used_fallback",
                      "price", "accommodates", "price_per_guest", "expected_ppg", "fence_ppg",
                      "price_gap", "price_gap_pct", "above_fence", "bedrooms", "amenities_count",
                      "peer_median_amenities", "status"]]
F8_status_counts.round(2).to_csv(OUT_DIR / "F8_status_counts.csv", index=False)
F8_listing_flags.round(2).to_csv(OUT_DIR / "F8_listing_flags.csv", index=False)
print("saved F8_status_counts.csv and F8_listing_flags.csv to", OUT_DIR.resolve())